In [10]:
import re
import xml.etree.ElementTree as ET

In [13]:
# Read primer_maple.mw as an XML file
tree = ET.parse('primer_maple.mw')
# tree = ET.parse('z.mw')
root = tree.getroot()

# Look for <Text-field> elements with the style attribute equal to "Text".  
# Nested inside those, look for <Equation> elements.  
# For all those elements, set the executable attribute to "false".
for text_field in root.findall('.//Text-field'):
    if text_field.get('style') == 'Text':
        for equation in text_field.findall('.//Equation'):
            equation.set('executable', 'false')
        # in the text content of the text_field, look for strings enclosed in `...`
        # For each such string, create a new <Font> element with the style 
        # attribute set to "Maple Input Placeholder" and the executable 
        # attribute set to "false", and replace the original string with this new element.
        text_content = text_field.text
        if text_content and '`' in text_content:
            # Find all occurrences of text enclosed in backticks
            parts = re.split(r'`([^`]*)`', text_content)
    
            # The very first element is always the text before the first backquote
            text_field.text = parts[0]
            
            # 3. Iterate through the rest of the list in pairs
            # parts[1::2] gets the backticked text, parts[2::2] gets the trailing text
            for i in range(1, len(parts), 2):
                backtick_text = parts[i]
                
                # Safely get the trailing text (in case of malformed strings)
                after_text = parts[i+1] if (i + 1) < len(parts) else ""
                
                # Create and append the new <Font> element
                e = ET.Element("Font", style="Maple Input Placeholder", executable="false")
                e.text = backtick_text
                e.tail = after_text
                text_field.append(e) 
                root.append(e)

# Save the modified XML back to a file
tree.write('primer_maple_fixed.mw')
# tree.write('zz.mw')